In [1]:
import sys
import torch
import socket
import importlib
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [2]:
import pandas as pd
import numpy as np
from src.config import raw_data_dir

data_path = raw_data_dir / "sample_data.npy"
sim = np.load(data_path, allow_pickle=True)

df = pd.DataFrame(sim)

In [3]:
df.head(10)

,initial_mass,initial_z,star_age,mass,logR,logP,logRho,logT,luminosity,opacity,x_mass_fraction_H,y_mass_fraction_He,z_mass_fraction_metals,eps_nuc,eps_nuc_neu_total,eps_grav_nh,eps_grav,zone,q
0,16.5,0.02,1.124803e+07,15.672543,2.769037,2.632306,-8.726443,3.565941,56927.325018,0.001937,0.667269,0.319678,0.013054,0.042915,1.739978e-29,-0.003933,-2.621206,1.0,1.0
1,16.5,0.02,1.124803e+07,15.672543,2.769037,2.632306,-8.726443,3.565941,56927.325018,0.001937,0.667269,0.319678,0.013054,0.042915,1.739978e-29,-0.003933,-2.621206,2.0,1.0
2,16.5,0.02,1.124803e+07,15.672543,2.769037,2.632306,-8.726443,3.565941,56927.325018,0.001937,0.667269,0.319678,0.013054,0.042915,1.739978e-29,-0.003933,-2.621206,3.0,1.0
3,16.5,0.02,1.124803e+07,15.672543,2.769037,2.632306,-8.726443,3.565941,56927.325018,0.001937,0.667269,0.319678,0.013054,0.042915,1.739978e-29,-0.003933,-2.621205,4.0,1.0
4,16.5,0.02,1.124803e+07,15.672543,2.769037,2.632306,-8.726443,3.565941,56927.325018,0.001937,0.667269,0.319678,0.013054,0.042915,1.739978e-29,-0.003933,-2.621205,5.0,1.0
5,16.5,0.02,1.124803e+07,15.672543,2.769037,2.632307,-8.726443,3.565941,56927.325018,0.001937,0.667269,0.319678,0.013054,0.042915,1.739978e-29,-0.003933,-2.621203,6.0,1.0
6,16.5,0.02,1.124803e+07,15.672543,2.769037,2.632307,-8.726443,3.565941,56927.325018,0.001937,0.667269,0.319678,0.013054,0.042915,1.739978e-29,-0.003933,-2.621202,7.0,1.0
7,16.5,0.02,1.124803e+07,15.672543,2.769037,2.632307,-8.726442,3.565941,56927.325018,0.001937,0.667269,0.319678,0.013054,0.042915,1.739979e-29,-0.003933,-2.621199,8.0,1.0
8,16.5,0.02,1.124803e+07,15.672543,2.769037,2.632307,-8.726442,3.565941,56927.325018,0.001937,0.667269,0.319678,0.013054,0.042915,1.739979e-29,-0.003933,-2.621196,9.0,1.0
9,16.5,0.02,1.124803e+07,15.672543,2.769037,2.632308,-8.726442,3.565941,56927.325018,0.001937,0.667269,0.319678,0.013054,0.042915,1.739980e-29,-0.003933,-2.621193,10.0,1.0


In [ ]:
importlib.reload(sys.modules["src.preprocessing"])
from src.preprocessing import check_missing_values, drop_constant_columns, fit_preprocess_scalers, rdp

print(check_missing_values(df))

df_reduced = drop_constant_columns(df)

df_scaled, scalers = fit_preprocess_scalers(
    df_reduced,
    normalize=True,
    standardize=False,
)

output_path = Path("data/processed/processed_data.npy")
output_path.parent.mkdir(parents=True, exist_ok=True)
np.save(output_path, df_scaled.to_numpy())

print(f"Saved processed dataset to {output_path}")

In [ ]:
df_scaled.head(10)

In [ ]:
import matplotlib.pyplot as plt

time = df_scaled['star_age'].to_numpy()[::10]
mass = df_scaled['mass'].to_numpy()[::10]
temp = df_scaled['logT'].to_numpy()[::10]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(projection='3d')

ax.plot_trisurf(time, mass, temp, cmap='viridis', edgecolor='none')
ax.set_xlabel('Star Age')
ax.set_ylabel('Mass')
ax.set_zlabel('Log T')
ax.set_title('Stellar Evolution Surface')
plt.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(
    data=[
        go.Mesh3d(
            x=time,
            y=mass,
            z=temp,
            intensity=temp,
            colorscale="Viridis",
            opacity=0.8,
        )
    ]
)
fig.update_layout(
    scene=dict(
        xaxis_title="Star Age",
        yaxis_title="Mass",
        zaxis_title="Log T",
    ),
    title="Stellar Evolution Surface",
)
fig.show()